# 🧠 Notebook 2 — Janelas, Níveis de Canal e Features

Parte 2 de 4.

Lê o **L1** do disco, recorta em **janelas de 4 s** (Nível 2), e extrai **handcrafted features** de forma compatível entre datasets, agregando por **região cerebral**.

### Por que agregar por região (e não por eletrodo fixo)?
Os datasets usam montagens diferentes: CHB-MIT é **bipolar**, Siena e Mendeley são **referenciais**, e o SeizeIT2 é **behind-the-ear**. Não existe “o mesmo eletrodo” entre todos. A solução que **funciona em qualquer montagem, dá vetor de tamanho fixo (para juntar datasets) e preserva interpretabilidade** é mapear cada canal para uma **região** (frontal/temporal/central/parietal/occipital) e agregar as features por região.

### Níveis de redução de canais = número de regiões
| Nível | Regiões | Análogo clínico | Datasets |
|---|---|---|---|
| **R5** | 5 (todas) | EEG hospitalar completo (~19 ch) | CHB-MIT, Siena, Mendeley |
| **R3** | 3 (frontal, temporal, central) | reduzido (~8 ch) | CHB-MIT, Siena, Mendeley |
| **R2** | 2 (frontal, temporal) | vestível (~4 ch) | CHB-MIT, Siena, Mendeley |
| **R1** | 1 (temporal) | ultra-compacto / vestível real | **+ SeizeIT2** |

O **SeizeIT2** (behind-the-ear ≈ temporal) só entra no nível **R1**, que é onde o seu formato físico se encaixa. A queda de desempenho de R5 → R1 é a **curva de degradação vestível** (resultado principal do TCC).

> **Atenção clínica:** crises focais começam numa região. Reduzir níveis pode remover a região do foco — e é exatamente isso que a curva de degradação **mede**. A planilha do Mendeley traz o foco de cada paciente, permitindo cruzar foco × regiões preservadas.

## 2.1 Imports e config

In [ ]:
import os, re, json, io, warnings, gc, html
import html.parser
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import pywt
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, welch, resample_poly
from scipy.stats  import kurtosis as sp_kurtosis, skew as sp_skew
from tqdm.auto    import tqdm

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
print("✅ Imports OK")

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Diretórios ───────────────────────────────────
ROOT_DIR    = 'data'
CHBMIT_DIR  = os.path.join(ROOT_DIR, 'chb-mit')
SIENA_DIR   = os.path.join(ROOT_DIR, 'siena')
SEIZEIT_DIR = os.path.join(ROOT_DIR, 'seizeit2')
MENDELEY_DIR= os.path.join(ROOT_DIR, 'mendeley')
L1_DIR      = os.path.join(ROOT_DIR, 'level1_signals')
L2_DIR      = os.path.join(ROOT_DIR, 'level2_windows')
FEAT_DIR    = os.path.join(ROOT_DIR, 'features')   # subpastas por nível: features/<level>/
RESULTS_DIR = os.path.join(ROOT_DIR, 'results')
for d in [CHBMIT_DIR, SIENA_DIR, SEIZEIT_DIR, MENDELEY_DIR,
          L1_DIR, L2_DIR, FEAT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Filtragem ──────────────────────────────────
F_HP, F_LP, F_ORDER = 0.5, 40.0, 4
NOTCH = {'CHBMIT': 60.0, 'Siena': 50.0, 'SeizeIT2': 50.0, 'Mendeley': 50.0}

# Reamostragem comum (Siena=512, Mendeley=500, demais=256) → fs único para
# tornar features espectrais comparáveis e permitir juntar datasets.
TARGET_FS = 256

# ── Janelamento ────────────────────────────────
WIN_SEC, OVERLAP = 4, 0.50

# ── Rotulagem ──────────────────────────────────
PRE_SEC, POST_SEC, IGAP_SEC = 10*60, 10*60, 10*60
LBL = dict(interictal=0, preictal=1, ictal=2, postictal=3, unknown=-1)
LBL_NAMES = {v: k for k, v in LBL.items()}

# ── Cap de interictal na extração (controla custo do SeizeIT2) ──────────
MAX_INTER_RATIO = 10   # máx. interictal:eventos por paciente (None desliga)

# ── PACIENTES POR DATASET ──────────────────────────────
PATIENTS = {
    'CHBMIT'  : ['chb01', 'chb02', 'chb03', 'chb04', 'chb05'],
    'Siena'   : ['PN00', 'PN01', 'PN03', 'PN05', 'PN06'],
    'Mendeley': ['p10', 'p11', 'p12', 'p13', 'p14'],
    'SeizeIT2': ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005'],
}
PILOT = {k: v[0] for k, v in PATIENTS.items()}
SEIZEIT_SESSION = 'ses-01'

# ── NÍVEIS DE REDUÇÃO DE CANAIS = subconjuntos de REGIÕES cerebrais ────────
# Por que regiões e não eletrodos fixos? Os datasets usam montagens diferentes
# (CHB-MIT é bipolar, Siena/Mendeley referencial, SeizeIT2 behind-the-ear), então
# "o mesmo eletrodo" não existe entre todos. Mapear cada canal → região e agregar
# por região (1) funciona em qualquer montagem, (2) dá vetor de tamanho fixo para
# poder juntar datasets, e (3) mantém a interpretabilidade (SHAP por região).
REGIONS = ['frontal', 'temporal', 'central', 'parietal', 'occipital']

LEVELS = {
    'R5': ['frontal', 'temporal', 'central', 'parietal', 'occipital'],  # completo (~19 ch)
    'R3': ['frontal', 'temporal', 'central'],                           # reduzido (~8 ch)
    'R2': ['frontal', 'temporal'],                                      # vestível (~4 ch)
    'R1': ['temporal'],                                                 # ultra-compacto / SeizeIT2
}
# Quais datasets participam de cada nível. SeizeIT2 (behind-the-ear, ~região
# temporal) só entra no nível mínimo R1, que é onde ele faz sentido físico.
LEVEL_DATASETS = {
    'R5': ['CHBMIT', 'Siena', 'Mendeley'],
    'R3': ['CHBMIT', 'Siena', 'Mendeley'],
    'R2': ['CHBMIT', 'Siena', 'Mendeley'],
    'R1': ['CHBMIT', 'Siena', 'Mendeley', 'SeizeIT2'],
}

# ── Mapa eletrodo → região (sistema 10-20; aceita variantes T7/T3 etc.) ────
ELECTRODE_REGION = {}
for _e in ['fp1','fp2','f7','f3','fz','f4','f8','af3','af4']:           ELECTRODE_REGION[_e]='frontal'
for _e in ['t3','t4','t5','t6','t7','t8','p7','p8','ft9','ft10','tp7','tp8']: ELECTRODE_REGION[_e]='temporal'
for _e in ['c3','cz','c4','fc1','fc2','cp1','cp2']:                     ELECTRODE_REGION[_e]='central'
for _e in ['p3','pz','p4','po3','po4']:                                 ELECTRODE_REGION[_e]='parietal'
for _e in ['o1','o2','oz']:                                             ELECTRODE_REGION[_e]='occipital'

print("✅ Configurações OK")
print(f"   fs alvo     : {TARGET_FS} Hz | Janela {WIN_SEC}s overlap {int(OVERLAP*100)}%")
print(f"   Pré/Pós     : {PRE_SEC//60}/{POST_SEC//60} min | cap interictal {MAX_INTER_RATIO}:1")
print(f"   Níveis      : " + " | ".join(f"{k}({len(v)} reg)" for k,v in LEVELS.items()))
for k, v in PATIENTS.items():
    print(f"   {k:9s}: {v}")

In [ ]:
def normalize_electrode(name):
    """Limpa um nome de canal: 'EEG Fp1-REF' -> 'fp1'. Para bipolar 'F7-T3'
    devolve o PRIMEIRO nó ('f7'), que define a região do canal."""
    s = str(name).lower()
    s = s.replace('eeg', ' ').replace('-ref', ' ').replace('-le', ' ').replace('ref', ' ')
    s = s.strip()
    # bipolar 'a-b' -> primeiro nó
    if '-' in s:
        s = s.split('-')[0]
    s = re.sub(r'[^a-z0-9]', '', s)
    return s

def channel_region(name, dataset=None):
    """Região cerebral de um canal. SeizeIT2 (behind-the-ear) -> temporal."""
    if dataset == 'SeizeIT2':
        return 'temporal'
    return ELECTRODE_REGION.get(normalize_electrode(name), None)

def regions_present(ch_names, dataset=None):
    """Dict região -> lista de índices de canais naquela região."""
    out = {r: [] for r in REGIONS}
    for i, nm in enumerate(ch_names):
        r = channel_region(nm, dataset)
        if r in out:
            out[r].append(i)
    return out

## 2.2 Janelamento → Nível 2 (com nomes dos canais)

Cada L2 guarda janelas, labels, tempos **e os nomes dos canais** (necessários para mapear canal→região).

In [ ]:
def extract_windows(data, labels, sfreq, win_sec=WIN_SEC, overlap=OVERLAP, min_pure=0.80):
    """(N, n_ch, ws) janelas + labels + tempos de início (s). Descarta unknown."""
    ws = int(win_sec * sfreq); step = max(1, int(ws * (1 - overlap)))
    wins, lbls, times = [], [], []
    for start in range(0, data.shape[1] - ws + 1, step):
        seg = labels[start:start+ws]
        if (seg == LBL['unknown']).any(): continue
        if (seg == LBL['ictal']).any():
            lv = LBL['ictal']
        else:
            uv, uc = np.unique(seg, return_counts=True); mi = np.argmax(uc)
            if uc[mi]/ws < min_pure: continue
            lv = int(uv[mi])
        wins.append(data[:, start:start+ws].astype(np.float32)); lbls.append(lv); times.append(start/sfreq)
    if not wins:
        return (np.empty((0, data.shape[0], ws), np.float32), np.empty(0, np.int8), np.empty(0))
    return np.stack(wins), np.array(lbls, np.int8), np.array(times)

In [ ]:
from collections import defaultdict

def save_level2(W, L, T, sfreq, ch_names, dataset, patient, out_dir=L2_DIR):
    key = f'{dataset}__{patient}'
    npz = os.path.join(out_dir, key + '_L2.npz')
    np.savez_compressed(npz, windows=W, labels=L, times=T,
                        sfreq=np.float32(sfreq), ch_names=np.array(ch_names), dataset=dataset)
    print(f"💾 L2: {os.path.basename(npz)}  ({len(W)} janelas, {W.shape[1]} canais)")
    return npz

# Agrupa os L1 por (dataset, paciente) e gera 1 L2 por paciente (com ch_names!)
l1_files = [os.path.join(L1_DIR, f) for f in os.listdir(L1_DIR) if f.endswith('_L1.npz')]
grouped = defaultdict(list)
for f in l1_files:
    base = os.path.basename(f).replace('_L1.npz','')
    ds, pat, _ = base.split('__', 2)
    grouped[(ds, pat)].append(f)

l2_index = {}
for (ds, pat), files in grouped.items():
    key = f'{ds}__{pat}'; out = os.path.join(L2_DIR, key + '_L2.npz')
    if os.path.exists(out):
        print(f"⏭️  Já existe: {os.path.basename(out)}"); l2_index[key]=out; continue
    Ws, Ls, Ts, ch_ref, sf_ref = [], [], [], None, None
    for f in files:
        npz = np.load(f, allow_pickle=True)
        W, L, T = extract_windows(npz['data'], npz['labels'], float(npz['sfreq']))
        if len(W):
            Ws.append(W); Ls.append(L); Ts.append(T)
            ch_ref = list(npz['ch_names']); sf_ref = float(npz['sfreq'])
    if Ws:
        l2_index[key] = save_level2(np.concatenate(Ws), np.concatenate(Ls),
                                    np.concatenate(Ts), sf_ref, ch_ref, ds, pat)
print(f"\n📦 Total L2: {len(l2_index)}")

## 2.3 Funções de feature (16 por canal)

In [ ]:
_BANDS = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,40)}

def temporal_feats(sig):
    d1 = np.diff(sig); act = float(np.var(sig))
    mob = float(np.sqrt(np.var(d1) / (np.var(sig) + 1e-10)))
    return np.array([float(np.std(sig)), float(np.var(sig)), float(np.sqrt(np.mean(sig**2))),
                     float(np.sum(np.abs(d1))), act, mob], dtype=np.float32)

def spectral_feats(sig, sfreq):
    nperseg = min(int(sfreq*4), max(int(sfreq), len(sig)//2))
    f, psd = welch(sig, fs=sfreq, nperseg=nperseg)
    bp = []
    for lo,hi in _BANDS.values():
        idx = (f>=lo)&(f<=hi); bp.append(float(np.trapz(psd[idx], f[idx])) if idx.sum()>1 else 0.0)
    pn = psd/(psd.sum()+1e-10); pn = pn[pn>0]
    return np.array(bp + [float(-np.sum(pn*np.log(pn)))], dtype=np.float32)

def dwt_energy_feats(sig, wavelet='db4', level=4):
    c = pywt.wavedec(sig.astype(np.float64), wavelet, level=level)
    return np.array([float(np.sum(x**2)) for x in c[1:level+1]], dtype=np.float32)

def channel_feats(sig, sfreq):
    """16 features de 1 canal: 6 temporais + 6 espectrais + 4 DWT."""
    return np.concatenate([temporal_feats(sig), spectral_feats(sig, sfreq), dwt_energy_feats(sig)])

N_FEAT_PER_REGION = 16
print("✅ Features por canal: 16 (6 temporais + 6 espectrais + 4 DWT)")

## 2.4 Pooling por região (vetor de tamanho fixo por nível)

In [ ]:
def region_pooled_features(W, ch_names, sfreq, level, dataset):
    """
    Para cada janela, agrega (média das features) os canais de cada REGIÃO do nível.
    Saída: (N, 16 * n_regioes_do_nivel). Tamanho fixo → datasets diferentes
    geram o MESMO vetor e podem ser juntados no treino. Região sem canal → zeros.
    Mantém identidade de região (ordem fixa) para interpretabilidade (SHAP).
    """
    regs = LEVELS[level]
    reg_idx = regions_present(ch_names, dataset)
    N = len(W); out = np.zeros((N, N_FEAT_PER_REGION * len(regs)), dtype=np.float32)
    for i in range(N):
        parts = []
        for r in regs:
            idxs = reg_idx.get(r, [])
            if idxs:
                feats = np.stack([channel_feats(W[i, ch].astype(np.float64), sfreq) for ch in idxs])
                parts.append(feats.mean(axis=0))     # pooling por região
            else:
                parts.append(np.zeros(N_FEAT_PER_REGION, dtype=np.float32))
        out[i] = np.concatenate(parts)
    return out

def feature_columns(level):
    """Nomes das colunas: <regiao>__<feat> — usado depois na interpretabilidade."""
    fnames = (['t_std','t_var','t_rms','t_linelen','t_hj_act','t_hj_mob'] +
              ['s_delta','s_theta','s_alpha','s_beta','s_gamma','s_specent'] +
              ['w_d1','w_d2','w_d3','w_d4'])
    return [f'{r}__{f}' for r in LEVELS[level] for f in fnames]

## 2.5 Cap de interictal + extração por nível

O cap limita o interictal a 10:1 antes de extrair (preserva eventos e o span temporal). As features são salvas em `features/<nível>/`.

In [ ]:
def cap_interictal(W, L, T, max_ratio=MAX_INTER_RATIO):
    """Limita interictal a max_ratio× eventos ANTES de extrair features (economia no
    SeizeIT2, ~95% interictal). Preserva eventos e o span temporal (FAR exato)."""
    if max_ratio is None: return W, L, T
    ii = np.where(L == LBL['interictal'])[0]
    ev = np.where(np.isin(L, [1,2,3]))[0]
    if len(ev)==0 or len(ii) <= max_ratio*len(ev): return W, L, T
    n_want = max_ratio*len(ev); stride = max(1, len(ii)//n_want)
    sel = ii[::stride][:n_want]
    sel = np.unique(np.concatenate([sel, ii[[0,-1]]]))   # mantém 1º e último → span exato
    allidx = np.sort(np.concatenate([ev, sel]))
    return W[allidx], L[allidx], T[allidx]

import time
# Extrai features POR NÍVEL. Salva em features/<level>/<dataset>__<patient>_{X,y,t}.npy
# Só processa o dataset em níveis onde ele participa (LEVEL_DATASETS).
feat_index = {lv: {} for lv in LEVELS}

for key, npz_path in l2_index.items():
    ds, pat = key.split('__', 2)[:2]
    npz = np.load(npz_path, allow_pickle=True)
    W_full = npz['windows']; L_full = npz['labels']
    T_full = npz['times'] if 'times' in npz.files else np.arange(len(W_full))*WIN_SEC*(1-OVERLAP)
    ch = list(npz['ch_names']); sf = float(npz['sfreq'])

    # cap de interictal (uma vez por paciente; mesmo subset serve a todos os níveis)
    n0 = len(W_full)
    W, L, T = cap_interictal(W_full, L_full, T_full)
    if len(W) < n0:
        print(f"✂️  {key}: interictal {n0} → {len(W)} (cap {MAX_INTER_RATIO}:1)")

    for lv in LEVELS:
        if ds not in LEVEL_DATASETS[lv]:
            continue
        outdir = os.path.join(FEAT_DIR, lv); os.makedirs(outdir, exist_ok=True)
        fx = os.path.join(outdir, key + '_X.npy')
        fy = os.path.join(outdir, key + '_y.npy')
        ft = os.path.join(outdir, key + '_t.npy')
        if os.path.exists(fx):
            feat_index[lv][key] = (fx, fy, ft); print(f"⏭️  [{lv}] já existe: {key}"); continue
        t0 = time.time()
        X = region_pooled_features(W, ch, sf, lv, ds)
        np.save(fx, X); np.save(fy, L); np.save(ft, T)
        feat_index[lv][key] = (fx, fy, ft)
        print(f"✅ [{lv}] {key}: {X.shape}  ({time.time()-t0:.0f}s)")

for lv in LEVELS:
    print(f"📦 Nível {lv}: {len(feat_index[lv])} pacientes  "
          f"({N_FEAT_PER_REGION*len(LEVELS[lv])} features)")

## 2.6 Undersampling (definido aqui, aplicado no treino do Notebook 3)

O balanceamento 1:3 mantém todas as janelas de evento e subsampla o interictal. **Só é aplicado no conjunto de treino** — o teste usa o paciente inteiro.

In [ ]:
def apply_ratio_multiclass(X, y, ratio=3, seed=RANDOM_SEED):
    """Mantém TODAS as janelas de evento (pré/ictal/pós) e subsampla interictal
    até ratio× o total de eventos. Usado APENAS no treino (no Notebook 3)."""
    ii = np.where(y == LBL['interictal'])[0]
    ev = np.where(np.isin(y, [1,2,3]))[0]
    if len(ev) == 0: return X[:0], y[:0]
    n_want = min(len(ii), ratio*len(ev))
    if n_want <= 0:
        sel = np.array([], dtype=int)
    else:
        stride = max(1, len(ii)//n_want); sel = ii[::stride][:n_want]
    idx = np.sort(np.concatenate([ev, sel]))
    return X[idx], y[idx]

# Demonstração do balanceamento em um paciente do nível R5
print("Exemplo de undersampling (treino) — ratio 1:3")
for lv in ['R5']:
    if feat_index[lv]:
        k0 = list(feat_index[lv])[0]
        X = np.load(feat_index[lv][k0][0]); y = np.load(feat_index[lv][k0][1])
        Xb, yb = apply_ratio_multiclass(X, y, 3)
        b = lambda a: {LBL_NAMES[int(v)]: int(c) for v,c in zip(*np.unique(a, return_counts=True))}
        print(f"  {k0}: antes={b(y)}")
        print(f"  {k0}: depois={b(yb)}")

---
✅ **Fim do Notebook 2.** Features por nível em `data/features/<nível>/`. Prossiga para o **Notebook 3 — Experimentos (LOSO + cenários)**.